In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/raw/india_agriculture_seed_sales_data.csv')


In [5]:
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score


df['Date']       = pd.to_datetime(df['Date'], format='%Y-%m')
df['Net_Profit'] = df['Total_Revenue'] - df['Total_Cost']

# Monthly aggregates 
monthly = df.groupby('Date').agg(
    Revenue      =('Total_Revenue','sum'),
    Cost         =('Total_Cost','sum'),
    Net_Profit   =('Net_Profit','sum'),
    Profit_Margin=('Profit_Margin_%','mean'),
    Rev_Growth   =('Revenue_Growth_%','mean'),
    Co_Share     =('Company_Market_Share_%','mean'),
    Comp_Share   =('Competitor_Market_Share_%','mean'),
    Units_Sold   =('Actual_Units_Sold','sum'),
    Days_to_Sell =('Days_to_Sell_Inventory','mean'),
    Farmer_Sent  =('Farmer_Sentiment_Score','mean'),
    Cost_Per_Unit=('Cost_Per_Unit','mean')
).reset_index().sort_values('Date').reset_index(drop=True)

monthly['t']         = np.arange(len(monthly))
monthly['month_num'] = monthly['Date'].dt.month
for k in range(1,4):
    monthly[f'sin_{k}'] = np.sin(2*np.pi*k*monthly['month_num']/12)
    monthly[f'cos_{k}'] = np.cos(2*np.pi*k*monthly['month_num']/12)

FEAT   = ['t','sin_1','cos_1','sin_2','cos_2','sin_3','cos_3']
X_hist = monthly[FEAT].values

# Future 6 months 
AHEAD        = 6
last_t       = monthly['t'].max()
last_date    = monthly['Date'].max()
future_dates = pd.date_range(start=last_date + pd.DateOffset(months=1), periods=AHEAD, freq='MS')

X_fut = []
for tv, m in zip(range(last_t+1, last_t+AHEAD+1), future_dates.month):
    row = [tv]
    for k in range(1,4): row.extend([np.sin(2*np.pi*k*m/12), np.cos(2*np.pi*k*m/12)])
    X_fut.append(row)
X_fut = np.array(X_fut)

def forecast(col, alpha=0.5):
    y = monthly[col].values
    m = Ridge(alpha=alpha).fit(X_hist, y)
    return m.predict(X_fut), r2_score(y, m.predict(X_hist))

rev_fc, rev_r2   = forecast('Revenue')
pm_fc,  pm_r2    = forecast('Profit_Margin')
rg_fc,  rg_r2    = forecast('Rev_Growth')
cos_fc, cos_r2   = forecast('Co_Share')
cmp_fc, cmp_r2   = forecast('Comp_Share')
us_fc,  us_r2    = forecast('Units_Sold')
dts_fc, dts_r2   = forecast('Days_to_Sell')
cpu_fc, cpu_r2   = forecast('Cost_Per_Unit')
fst_fc, fst_r2   = forecast('Farmer_Sent')

#  Print table 
print('='*75)
print('GREEN HARVEST SEEDS — 6-MONTH FORECAST (Jan–Jun 2027)')
print('='*75)
print(f'{"Month":<12}{"Rev(₹M)":>9}{"Margin%":>9}{"Growth%":>9}{"OurSh%":>9}{"Units(K)":>9}{"DtS":>7}{"CPU₹":>8}')
print('-'*75)
for i,d in enumerate(future_dates):
    print(f'{d.strftime("%b-%Y"):<12}{rev_fc[i]/1e6:>9.2f}{pm_fc[i]:>9.2f}'
          f'{rg_fc[i]:>9.2f}{cos_fc[i]:>9.1f}{us_fc[i]/1000:>9.1f}'
          f'{dts_fc[i]:>7.1f}{cpu_fc[i]:>8.2f}')

print(f'\nModel R² — Rev:{rev_r2:.3f} Margin:{pm_r2:.3f} Growth:{rg_r2:.3f} '
      f'Share:{cos_r2:.3f} Units:{us_r2:.3f}')

#  Alerts ─
print('\n🚨 FORECAST ALERTS:')
for i,d in enumerate(future_dates):
    if pm_fc[i] < 0:   print(f'  ⛔ {d.strftime("%b-%Y")}: NEGATIVE MARGIN ({pm_fc[i]:.2f}%)')
    elif pm_fc[i]<1.5: print(f'  ⚠  {d.strftime("%b-%Y")}: Below 1.5% critical ({pm_fc[i]:.2f}%)')
    if rg_fc[i] < -15: print(f'  🔴 {d.strftime("%b-%Y")}: Revenue growth extremely negative ({rg_fc[i]:.2f}%)')


#  2×3 Forecast dashboard ────────────────────────────────
fig = plt.figure(figsize=(20,13))
gs  = gridspec.GridSpec(2,3, figure=fig, hspace=0.48, wspace=0.38)

configs = [
    ('Revenue (₹M)',        monthly['Revenue']/1e6,    rev_fc/1e6, 'steelblue', rev_r2),
    ('Profit Margin %',     monthly['Profit_Margin'],   pm_fc,      '#2ca02c',   pm_r2),
    ('Revenue Growth %',    monthly['Rev_Growth'],      rg_fc,      '#d62728',   rg_r2),
    ('Our Market Share %',  monthly['Co_Share'],        cos_fc,     'teal',      cos_r2),
    ('Units Sold (K)',       monthly['Units_Sold']/1000, us_fc/1000, 'purple',    us_r2),
    ('Days to Sell Inv.',   monthly['Days_to_Sell'],   dts_fc,     '#ff7f0e',   dts_r2),
]
for (r,c), (title, hist, fc, color, r2) in zip([(0,0),(0,1),(0,2),(1,0),(1,1),(1,2)], configs):
    ax = fig.add_subplot(gs[r,c])
    ax.plot(monthly['Date'], hist, color=color, lw=2, label='Historical', alpha=0.8)
    ax.plot(future_dates, fc,  color=color, lw=2.5, ls='--', marker='o', ms=5, label='Forecast')
    ax.fill_between(future_dates, fc*0.95, fc*1.05, alpha=0.18, color=color, label='±5% CI')
    ax.axvspan(future_dates[0], future_dates[-1], alpha=0.06, color='gray')
    if 'Margin' in title: ax.axhline(1.5, color='red', lw=1, ls=':', alpha=0.8)
    if 'Growth' in title: ax.axhline(0,   color='black', lw=0.8, ls='--', alpha=0.6)
    ax.set_title(f'{title}\n(R²={r2:.3f})', fontsize=10)
    ax.set_ylabel(title, fontsize=8)
    ax.legend(fontsize=7); ax.tick_params(axis='x', rotation=30, labelsize=7)

fig.suptitle('GREEN HARVEST SEEDS — 6-Month Forecast: Jan–Jun 2027\n(Trend + Fourier Seasonality · Ridge Regression)',
             fontsize=13, fontweight='bold', y=1.01)
plt.savefig('../outputs/figures/time_series_forecast.png', bbox_inches='tight', dpi=140)
print('Saved: time_series_forecast.png')
plt.close()

#  Margin + Cost trajectory detail 
fig, axes = plt.subplots(1, 2, figsize=(16,5))

hist_pm = monthly['Profit_Margin']
axes[0].plot(monthly['Date'], hist_pm, 'g-', lw=2, label='Historical')
axes[0].plot(future_dates, pm_fc, 'g--o', lw=2.5, ms=7, label='Forecast Jan–Jun 2027')
axes[0].fill_between(future_dates, pm_fc-1, pm_fc+1, alpha=0.2, color='green', label='±1% band')
axes[0].axhline(3,   color='orange', lw=1.5, ls='--', label='3% warning')
axes[0].axhline(1.5, color='red',    lw=1.5, ls=':',  label='1.5% critical')
axes[0].axhline(0,   color='black',  lw=0.8, ls='--', alpha=0.5)
axes[0].axvspan(future_dates[0], future_dates[-1], alpha=0.08, color='gray', label='Forecast window')
for i,d in enumerate(future_dates):
    axes[0].annotate(f'{pm_fc[i]:.1f}%', (d, pm_fc[i]),
                     textcoords='offset points', xytext=(0,9), ha='center', fontsize=8, color='darkgreen')
axes[0].set_title('Profit Margin % — Historical & 6-Month Forecast (Jan–Jun 2027)')
axes[0].set_ylabel('Profit Margin %'); axes[0].legend(fontsize=7)

axes[1].plot(monthly['Date'], monthly['Cost_Per_Unit'], '#d62728', lw=2, label='Cost Per Unit ₹')
axes[1].plot(future_dates, cpu_fc, '#d62728', lw=2.5, ls='--', marker='o', ms=7, label='Forecast')
axes[1].plot(monthly['Date'], monthly['Revenue']/monthly['Units_Sold'],
             'steelblue', lw=2, alpha=0.8, label='Avg Revenue/Unit ₹')
for i,d in enumerate(future_dates):
    axes[1].annotate(f'₹{cpu_fc[i]:.0f}', (d, cpu_fc[i]),
                     textcoords='offset points', xytext=(0,9), ha='center', fontsize=8, color='#d62728')
axes[1].axvspan(future_dates[0], future_dates[-1], alpha=0.08, color='gray')
axes[1].set_title('Cost Per Unit Inflation vs Revenue Per Unit\n(6.5%/yr cost CAGR squeezing all margin)')
axes[1].set_ylabel('₹ per unit'); axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig('../outputs/figures/forecast_detail.png', bbox_inches='tight', dpi=140)
print('Saved:forecast_detail.png')
plt.close()

print('\n✅ Time Series Forecasting Complete')


GREEN HARVEST SEEDS — 6-MONTH FORECAST (Jan–Jun 2027)
Month         Rev(₹M)  Margin%  Growth%   OurSh% Units(K)    DtS    CPU₹
---------------------------------------------------------------------------
Jan-2027        56.05    -0.18   -17.58     42.3    262.1   65.6  212.68
Feb-2027        55.52    -0.79   -18.62     42.5    255.1   65.7  216.42
Mar-2027        55.78    -0.64   -18.43     42.5    258.3   65.5  215.12
Apr-2027        56.85    -0.49   -18.16     42.3    265.9   65.2  213.67
May-2027        57.04    -0.66   -18.36     42.3    265.9   65.2  214.46
Jun-2027        56.43    -0.67   -18.42     42.5    262.4   65.2  214.66

Model R² — Rev:0.832 Margin:0.976 Growth:0.925 Share:0.064 Units:0.225

🚨 FORECAST ALERTS:
  ⛔ Jan-2027: NEGATIVE MARGIN (-0.18%)
  🔴 Jan-2027: Revenue growth extremely negative (-17.58%)
  ⛔ Feb-2027: NEGATIVE MARGIN (-0.79%)
  🔴 Feb-2027: Revenue growth extremely negative (-18.62%)
  ⛔ Mar-2027: NEGATIVE MARGIN (-0.64%)
  🔴 Mar-2027: Revenue growth extre